# Real World Classification

## Load data

Import the necessary libraries

In [ ]:
# If you do not use colab. You should install these packages.
# !pip install numpy
# !pip install pandas
# !pip install matplotlib
# !pip install scikit-learn
# !pip install graphviz

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

seed=40
np.random.seed(seed)

load the data

In [9]:
# Load data from the regularization demo CSV
df = pd.read_csv('data/NYCU_Iris.csv')
df.head()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,BranchLength,AvgDust,SepalLengthMajorAxis,SepalLengthMinorAxis,SepalLengthElongation,...,LeafHueVariation,CanopyLightCapture,CanopyMoistureSignal,SoilMoistureSignal,AmbientHumiditySignal,ThermalResponseIndex,UVExposureIndex,WindStressIndex,GrowthVigorIndex,Species
0,1,7.0,NaN,4.700000,1.4,16.9,37.5,6.769877,6.755471,6.964281,...,0.466458,0.961982,-0.041964,-2.573335,-0.290155,1.346631,0.334072,0.303819,-1.318415,Iris-versicolor
1,2,6.4,3.2,4.500000,1.5,16.4,95.1,6.202714,6.129652,6.201464,...,-1.270085,1.283433,-1.155498,-0.295384,0.684086,-0.426519,0.113067,1.562539,0.095345,Iris-versicolor
2,3,6.9,NaN,4.900000,1.5,16.9,73.2,6.726317,6.762992,6.784647,...,0.562054,1.238704,0.345857,1.172525,0.320302,-0.563687,-1.514457,-0.179045,-0.577574,Iris-versicolor
3,4,5.5,2.3,5.085612,1.3,15.6,59.9,5.608292,5.529096,5.382254,...,-0.165898,0.914796,1.252364,-0.301381,-0.466333,-0.773040,-0.793858,0.483426,1.210299,Iris-versicolor
4,5,6.5,2.8,4.600000,1.5,16.4,15.6,6.300774,6.443553,6.274360,...,-0.602098,2.045948,-0.465691,0.862299,2.085633,2.047018,0.005283,0.506305,0.766709,Iris-versicolor


## Data Preprocessing

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import KNNImputer

def data_preprocessing(df):
    # transform label to bi-class
    df['Species'] = df['Species'].astype(str).str.strip()
    le = LabelEncoder()
    df['Species'] = le.fit_transform(df['Species'])

    feature_cols = [c for c in df.columns if c not in ['Id', 'Species']]

    # transform string to number
    for col in feature_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        
    # TODO: Replace the missing values using “Nearest Neighbors Imputation”
    # ---------- Start your code below ----------
    missing_cols = df[feature_cols].columns[df[feature_cols].isnull().any()].tolist()

    print("Columns with missing values before imputation:")
    for col in missing_cols:
        print(f"{col}: median = {df[col].median():.4f}, std = {df[col].std():.4f}")

    imputer = KNNImputer(n_neighbors=9)
    df[feature_cols] = imputer.fit_transform(df[feature_cols])

    print("\nColumns with missing values after KNN (N=7) imputation:")
    for col in missing_cols:
        print(f"{col}: median = {df[col].median():.4f}, std = {df[col].std():.4f}")
    
    # ---------- --------------------- ----------
     
    return df, feature_cols

df, feature_cols = data_preprocessing(df)

Columns with missing values before imputation:
SepalLengthCm: median = 6.3000, std = 1.0371
SepalWidthCm: median = 2.9000, std = 0.3896
PetalLengthCm: median = 5.0856, std = 1.5828
PetalWidthCm: median = 1.6000, std = 0.7067
BranchLength: median = 16.3000, std = 1.0352

Columns with missing values after KNN (N=7) imputation:
SepalLengthCm: median = 6.3000, std = 1.0089
SepalWidthCm: median = 2.8571, std = 0.3722
PetalLengthCm: median = 5.0041, std = 1.4669
PetalWidthCm: median = 1.6929, std = 0.6942
BranchLength: median = 16.3000, std = 1.0110


In [ ]:
df.describe()

## Data Exploration

In [ ]:
from sklearn.feature_selection import r_regression

# TODO: Complete the 4. Data Exploration

plt.figure(figsize=(8, 5))
plt.hist(df['PetalWidthCm'], bins=15, edgecolor='black')
plt.title('Histogram of PetalWidthCm')
plt.xlabel('PetalWidthCm')
plt.ylabel('Frequency')
plt.show()

exclude_cols = ['PetalWidthCm', 'Species', 'Id']
candidate_features = [col for col in df.columns if col not in exclude_cols]

X = df[candidate_features]
y = df['PetalWidthCm']

corr_values = r_regression(X, y)
corr_series = pd.Series(corr_values, index=candidate_features)

largest_positive_feature = corr_series.idxmax()
largest_positive_value = corr_series.max()

print("Feature with the largest positive correlation with PetalWidthCm:")
print(f"{largest_positive_feature}: {largest_positive_value:.4f}")

top5_negative = corr_series.sort_values().head(5)

print("\nTop 5 features with the strongest negative correlations with PetalWidthCm:")
print(top5_negative)

In [ ]:
# feature from 4(b)
largest_positive_feature = "PetalWidthCompactness"

# top 5 negative features from 4(c)
top5_negative_features = [
    "SepalWidthMajorAxis",
    "SepalGlossIndex",
    "SepalWidthCompactness",
    "SepalWidthCurvature",
    "SepalWidthMinorAxis"
]

# combine all features for boxplot
selected_features = [largest_positive_feature] + top5_negative_features

plt.figure(figsize=(12, 6))
df[selected_features].boxplot()
plt.title("Boxplot of Features Identified in 4(b) and 4(c)")
plt.ylabel("Value")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Model Training

### Prepare the data

In [ ]:
# Split the data into training and testing sets
from sklearn.model_selection import train_test_split

# normalize the data to [0,1]
for col in feature_cols:
    col_min = df[col].min()
    col_max = df[col].max()
    if col_max > col_min:
        df[col] = (df[col] - col_min) / (col_max - col_min)
    else:
        df[col] = 0.0
        
X = df[feature_cols].values.astype(float)
y = df['Species'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=seed)

df.head()

### Train the model!

In [ ]:
# Use the LinearModel to fit the data

from model.linear_model import LinearModel
from model.metrics import logloss
from model.gradients import logloss_sigmoid_grad
from model.utils import *
from model.activations import sigmoid

# Model configuration
loss_fn = logloss
act_fn = sigmoid
grad_fn = logloss_sigmoid_grad

np.random.seed(seed)
l2 = 0.01
model = LinearModel(dim=X_train.shape[1], is_reg=False, loss_fn=loss_fn, act_fn=act_fn, grad_fn=grad_fn)
model.fit(X_train, y_train,lr=0.1, n_iteration =10000,val_ratio=0.2, reg_type='l2', reg_lambda=l2)

# print model parameters
print("Model parameters (weights):", model.W)
# sum of absolute values of weights
print("Sum of absolute values of weights:", np.sum(np.abs(model.W)))

## Metrics

In [ ]:
# use evaluate_binary_classifier to evaluate the model on the test set
from model.metrics import evaluate_binary_classifier

y_pred = model.predict(X_test)
evaluate_binary_classifier(y_test, y_pred)